Does the candidate set keep getting richer past K = 15? **GPU T4 x2**, Internet on.

Attach `prepare-data-for-word-reranker` under Add Input → Your Work → Notebook Output.

In [ ]:
REPO_URL = "https://github.com/Splestule/candidate_reranker.git"
BRANCH = "main"
K = 30

In [ ]:
import subprocess, sys
from pathlib import Path

CODE = Path("/kaggle/working/candidate_reranker")
if not (CODE / ".git").exists():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(CODE)],
                   check=True)

sys.path.insert(0, str(CODE / "src"))
import kaggle_env as K_env

COMMIT = K_env.sync(REPO_URL, BRANCH)
K_env.gpu_info()
env = K_env.prepare(COMMIT)

K = 30 needs twice the activation memory of K = 15. If this cell reports out of memory, lower `K` to 24 and rerun from here.

In [ ]:
rc = K_env.run(env, "selftest.py",
               "--base_model", env.base_model,
               "--adapter", env.adapter,
               "--librispeech", env.librispeech / "test-clean",
               "--n_utts", "3",
               "--n_candidates", K)
assert rc == 0, f"selftest exit code {rc}, see the output above"

Roughly 4 hours for test-clean at K = 30. `--resume` picks up if the session dies.

In [ ]:
K_env.run(env, "dump_candidates.py",
          "--source", "librispeech", "--path", env.librispeech / "test-clean",
          "--base_model", env.base_model, "--adapter", env.adapter,
          "--n_candidates", K,
          "--out", env.results / f"test-clean-k{K}-{COMMIT}.jsonl",
          "--tag", "test-clean", "--resume")

First look. The ladder is derived from this one dump, so every rung sees the same utterances.

In [ ]:
DUMP = env.results / f"test-clean-k{K}-{COMMIT}.jsonl"

K_env.run(env, "analyze.py", DUMP, "--json", env.results / f"test-clean-k{K}-{COMMIT}.stats.json")

for k in [5, 10, 15, 20, 25, K]:
    print("\n" + "#" * 74 + f"\n# composition at k = {k}\n" + "#" * 74)
    K_env.run(env, "analyze_compose.py", DUMP,
              "--max_k", k, "--alpha", 0.5, "--eps_conf", 0.7, "--gamma", 1.0,
              "--n_boot", 1000,
              "--json", env.results / f"compose-k{k}-{COMMIT}.json")